In [ ]:
from langchain_groq import ChatGroq
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableBranch, RunnableLambda
from pydantic import BaseModel, Field
from typing import Literal
from dotenv import load_dotenv, find_dotenv

In [ ]:
load_dotenv(find_dotenv())

model = ChatGroq(model="openai/gpt-oss-120b", temperature=0, max_tokens=512)

In [ ]:
class Response(BaseModel):
    sentiment:Literal["positive", "negative"] = Field(description="Give the sentiment of the feedback")

parser1 = PydanticOutputParser(pydantic_object=Response)
parser2 = StrOutputParser()

In [ ]:
prompt1 = PromptTemplate(
    template = "Analyze and give sentiment of this feedback: {feedback}\n\n\n{format_instructions}",
    input_variables=["feedback"],
    partial_variables={"format_instructions": parser1.get_format_instructions()}
)

prompt2 = PromptTemplate(
    template='Write an appropriate response to this positive feedback \n {feedback}',
    input_variables=['feedback']
)

prompt3 = PromptTemplate(
    template='Write an appropriate response to this negative feedback \n {feedback}',
    input_variables=['feedback']
)

In [ ]:
classifier_chain = prompt1 | model | parser1

branch_chain = RunnableBranch(
    (lambda x: x.sentiment=="positive", prompt2 | model | parser2),
    (lambda x: x.sentiment=="negative", prompt3 | model | parser2),
    RunnableLambda(lambda x: "could not find sentiment")
)

chain = classifier_chain | branch_chain

In [ ]:
chain.invoke({"feedback":"I am very disappointed with this phone. The battery drains quickly, the performance is slow, and the phone sometimes hangs while using basic applications. The camera quality is also not as good as expected. Overall, the quality does not justify the price, and I am unhappy with my purchase. I would not recommend this phone to others."})

'**Subject: We’re Sorry to Hear About Your Experience – Let’s Make It Right**\n\nHi\u202f[Customer’s Name],\n\nThank you for taking the time to share your feedback. I’m truly sorry to hear that your recent experience with us fell short of the high standards we strive to uphold.\n\nI understand how frustrating it can be when things don’t work as expected, and I want to assure you that we take your concerns seriously. To help us resolve this promptly, could you please provide a few more details about what happened (e.g., the product/service involved, the date of the incident, and any reference numbers you may have)? This information will allow us to investigate the issue thoroughly and prevent it from happening again.\n\nIn the meantime, here’s what we’d like to do for you:\n\n1. **Immediate assistance** – Our support team is ready to address the problem right away.  \n2. **Compensation/Replacement** – Depending on the situation, we can offer a refund, replacement, or a credit toward a f